# Patent Phrase Matching - DeBERTa Training
**Google Colab 실행용**

⚠️ Runtime → Change runtime type → **A100 / L4 GPU** 선택 후 실행 (bfloat16 지원 GPU 권장)

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/aicodinggym_2/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/aicodinggym_2/hf_cache',    exist_ok=True)
print('디렉토리 준비 완료')

In [ ]:
# 3. 데이터 확인
import os
data_path = '/content/drive/MyDrive/aicodinggym_2/us-patent-phrase-to-phrase-matching.zip'
if os.path.exists(data_path):
    print(f'✅ 데이터 확인: {data_path}')
else:
    print(f'❌ 데이터 없음! 현재 Drive 경로: {data_path}')

In [ ]:
# 5. 스크립트 다운로드 & 경로 패치
import re, shutil

# 캐시 무력화를 위해 임의 파라미터 추가
import time
!wget -q -O /content/deberta_finetune.py \
    'https://raw.githubusercontent.com/castlhoo/DSC204_us-patent-phrase-to-phrase-matching/main/deberta_finetune.py'

with open('/content/deberta_finetune.py', 'r') as f:
    code = f.read()

# 경로 패치
code = re.sub(
    r'"data_path"\s*:\s*"data/us-patent-phrase-to-phrase-matching\.zip"',
    '"data_path"          : "/content/drive/MyDrive/aicodinggym_2/us-patent-phrase-to-phrase-matching.zip"',
    code
)
code = re.sub(
    r'"ckpt_dir"\s*:\s*"checkpoints"',
    '"ckpt_dir"           : "/content/drive/MyDrive/aicodinggym_2/checkpoints"',
    code
)

# torch.load weights_only 패치 (PyTorch 2.6+)
code = code.replace(
    'torch.load(step_ckpt_path, map_location=CFG["device"])',
    'torch.load(step_ckpt_path, map_location=CFG["device"], weights_only=False)'
).replace(
    'torch.load(epoch_ckpt_path, map_location=CFG["device"])',
    'torch.load(epoch_ckpt_path, map_location=CFG["device"], weights_only=False)'
).replace(
    'torch.load(best_ckpt_path, map_location=CFG["device"])',
    'torch.load(best_ckpt_path, map_location=CFG["device"], weights_only=False)'
)

# RNG 상태 복원 패치 (CUDA tensor → CPU)
code = code.replace(
    "torch.set_rng_state(rng['torch'])",
    "torch.set_rng_state(rng['torch'].cpu() if hasattr(rng['torch'], 'cpu') else rng['torch'])"
)

with open('/content/deberta_finetune.py', 'w') as f:
    f.write(code)

print('✅ 경로 패치 완료')
!grep -n 'data_path\|ckpt_dir\|bfloat16\|GradScaler' /content/deberta_finetune.py | head -10

In [ ]:
# 5. 스크립트 다운로드 & 경로 패치
import re

!wget -q -O /content/deberta_finetune.py \
    'https://raw.githubusercontent.com/castlhoo/DSC204_us-patent-phrase-to-phrase-matching/main/deberta_finetune.py'

with open('/content/deberta_finetune.py', 'r') as f:
    code = f.read()

# 경로 패치
code = re.sub(
    r'"data_path"\s*:\s*"data/us-patent-phrase-to-phrase-matching\.zip"',
    '"data_path"          : "/content/drive/MyDrive/aicodinggym_2/us-patent-phrase-to-phrase-matching.zip"',
    code
)
code = re.sub(
    r'"ckpt_dir"\s*:\s*"checkpoints"',
    '"ckpt_dir"           : "/content/drive/MyDrive/aicodinggym_2/checkpoints"',
    code
)
code = code.replace(
    'os.environ["HF_HOME"]  = os.path.expanduser("~/.hf_cache")',
    'os.environ["HF_HOME"]  = "/content/drive/MyDrive/aicodinggym_2/hf_cache"'
)

with open('/content/deberta_finetune.py', 'w') as f:
    f.write(code)

print('✅ 경로 패치 완료')
!grep -n 'data_path\|ckpt_dir\|HF_HOME' /content/deberta_finetune.py | head -5

In [ ]:
# 6. 학습 시작 (출력이 실시간으로 나옵니다)
!python /content/deberta_finetune.py